# MIMIC Ventilator Weaning Prediction with Trajectory Features
Compare baseline, summary statistics, trajectory, and combined features across models.

## Setup

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

sys.path.append(os.path.abspath('..'))
from notebook_utils import biomarker_summary_stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ Imports successful')

✓ Imports successful


## Load Data

In [2]:
pred_with_probs_path = '../../../results/mimic/ventilator/ventilator_prediction_dataset_with_probs.csv'
outcome_path = '../../../results/mimic/ventilator/ventilator_outcomes.csv'
ts_path = '../../../results/mimic/ventilator/pf_ratio_timeseries.csv'

df = pd.read_csv(pred_with_probs_path)
outcomes = pd.read_csv(outcome_path)
pf_ts = pd.read_csv(ts_path)

print(f'✓ Loaded prediction dataset with probs: {len(df):,} rows')
print(f'✓ Outcomes: {len(outcomes):,} rows')
print(f'✓ P/F ratio TS: {len(pf_ts):,} rows')

df = df.merge(outcomes[['hadm_id', 'time_day', 'target_weaning_success']], on=['hadm_id', 'time_day'], how='left')
df = df[df['target_weaning_success'].notna()].copy()
df['target_weaning_success'] = df['target_weaning_success'].astype(int)

if 'prob_improving' not in df.columns and {'prob_gradual_improvement', 'prob_rapid_improvement'}.issubset(df.columns):
    df['prob_improving'] = df['prob_gradual_improvement'].fillna(0) + df['prob_rapid_improvement'].fillna(0)

traj_cols = [c for c in df.columns if c.startswith('prob_') or c.endswith('_stable') or c.endswith('_gradual') or c.endswith('_rapid') or 'worsening' in c or 'improving' in c]

print(f'Final dataset rows: {len(df):,}')
print(f'Outcome rate: {df["target_weaning_success"].mean():.1%}')
print(f'Trajectory columns: {len(traj_cols)}')

✓ Loaded prediction dataset with probs: 278,429 rows
✓ Outcomes: 24,338 rows
✓ P/F ratio TS: 58,353 rows
Final dataset rows: 24,336
Outcome rate: 6.3%
Trajectory columns: 4


## Feature Sets

In [3]:
pf_summary = biomarker_summary_stats(pf_ts, value_col='pf_ratio', lookback_days=3)
df = df.merge(pf_summary, on=['hadm_id', 'time_day'], how='left')

exclude_cols = {'hadm_id', 'time_day', 'charttime', 'admittime', 'dischtime', 'target_weaning_success'}
numeric_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
traj_cols = [c for c in numeric_cols if c.startswith('prob_') or c.endswith('_stable') or c.endswith('_gradual') or c.endswith('_rapid') or 'worsening' in c or 'improving' in c]
summary_cols = [c for c in numeric_cols if c.endswith('_3d') or '_trend_' in c or '_change_' in c]
base_cols = [c for c in numeric_cols if c not in traj_cols and c not in summary_cols]

static_name_tokens = ['age', 'gender', 'sex', 'baseline', 'admit', 'admission', 'ethnicity', 'race', 'height', 'weight', 'bmi']
static_cols = [c for c in base_cols if any(tok in c.lower() for tok in static_name_tokens)]
dynamic_cols = [c for c in base_cols if c not in static_cols]

feature_sets = {
    'Trajectory Only': traj_cols,
    'Summary Stats Only': summary_cols,
    'Trajectory + Summary Stats': traj_cols + summary_cols,
    'Static Only': static_cols,
    'Trajectory + Static': traj_cols + static_cols,
    'Summary Stats + Static': summary_cols + static_cols,
    'Trajectory + Summary Stats + Static': traj_cols + summary_cols + static_cols,
}

if len(dynamic_cols) > 0:
    feature_sets['Static + Dynamic'] = static_cols + dynamic_cols
    feature_sets['Trajectory + Static + Dynamic'] = traj_cols + static_cols + dynamic_cols
    feature_sets['Summary + Static + Dynamic'] = summary_cols + static_cols + dynamic_cols
    feature_sets['Trajectory + Summary + Static + Dynamic'] = traj_cols + summary_cols + static_cols + dynamic_cols

for name, cols in feature_sets.items():
    print(f'{name}: {len(cols)} features')

 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number 

## Model Comparison

In [5]:
df = df.drop_duplicates(subset=['hadm_id', 'time_day'])

In [6]:
df.shape

(20815, 102)

In [ ]:
models_to_evaluate = {
    'LogReg': lambda: LogisticRegression(max_iter=300, n_jobs=-1),
    'RF': lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=920),
    'HGB': lambda: HistGradientBoostingClassifier(random_state=920),
    'XGB': lambda: XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, tree_method='hist', n_jobs=-1, eval_metric='logloss', random_state=920),
}

n_repeats = 10
n_splits = 5
results = []
y = df['target_weaning_success']
groups = df['hadm_id']

for model_name, model_fn in models_to_evaluate.items():
    dataset_model = df.copy()
    if len(traj_cols) > 0:
        dataset_model[traj_cols] = dataset_model.groupby('hadm_id')[traj_cols].ffill(limit=2)
    if model_name not in ['XGB', 'HGB']:
        if len(traj_cols) > 0:
            dataset_model[traj_cols] = dataset_model[traj_cols].fillna(0)
        if len(summary_cols) > 0:
            dataset_model[summary_cols] = dataset_model[summary_cols].fillna(0)

    for feature_set_name, feature_cols in feature_sets.items():
        if len(feature_cols) == 0:
            continue
        aucs, auprcs = [], []
        for repeat in range(n_repeats):
            shuffle_idx = np.random.RandomState(seed=920 + repeat).permutation(len(dataset_model))
            dataset_repeat = dataset_model.iloc[shuffle_idx].reset_index(drop=True)
            y_repeat = y.iloc[shuffle_idx].reset_index(drop=True)
            groups_repeat = groups.iloc[shuffle_idx].reset_index(drop=True)
            gkf = GroupKFold(n_splits=n_splits)
            for train_idx, test_idx in gkf.split(dataset_repeat, y_repeat, groups_repeat):
                X_train = dataset_repeat.iloc[train_idx][feature_cols]
                X_test = dataset_repeat.iloc[test_idx][feature_cols]
                y_train = y_repeat.iloc[train_idx]
                y_test = y_repeat.iloc[test_idx]

                if model_name not in ['XGB', 'HGB']:
                    imputer = SimpleImputer(strategy='median')
                    X_train_imputed = imputer.fit_transform(X_train)
                    X_test_imputed = imputer.transform(X_test)
                else:
                    X_train_imputed = X_train
                    X_test_imputed = X_test

                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_imputed)
                X_test_scaled = scaler.transform(X_test_imputed)

                model = model_fn()
                model.fit(X_train_scaled, y_train)

                if hasattr(model, 'predict_proba'):
                    probs = model.predict_proba(X_test_scaled)[:, 1]
                else:
                    probs = model.decision_function(X_test_scaled)

                aucs.append(roc_auc_score(y_test, probs))
                auprcs.append(average_precision_score(y_test, probs))

        print(f"{model_name} | {feature_set_name}: AUROC {np.mean(aucs):.3f} ± {np.std(aucs):.3f}, AUPRC {np.mean(auprcs):.3f} ± {np.std(auprcs):.3f}")
        results.append({
            'model': model_name,
            'feature_set': feature_set_name,
            'auroc_mean': np.mean(aucs),
            'auroc_std': np.std(aucs),
            'auprc_mean': np.mean(auprcs),
            'auprc_std': np.std(auprcs),
        })

results_df = pd.DataFrame(results)
results_df

LogReg | Trajectory Only: AUROC 0.638 ± 0.007, AUPRC 0.103 ± 0.003
LogReg | Summary Stats Only: AUROC 0.741 ± 0.021, AUPRC 0.228 ± 0.030
LogReg | Trajectory + Summary Stats: AUROC 0.749 ± 0.019, AUPRC 0.232 ± 0.027
LogReg | Static Only: AUROC 0.618 ± 0.008, AUPRC 0.117 ± 0.004
LogReg | Trajectory + Static: AUROC 0.691 ± 0.009, AUPRC 0.148 ± 0.005
LogReg | Summary Stats + Static: AUROC 0.744 ± 0.018, AUPRC 0.230 ± 0.028
LogReg | Trajectory + Summary Stats + Static: AUROC 0.753 ± 0.017, AUPRC 0.234 ± 0.025
LogReg | Static + Dynamic: AUROC 0.820 ± 0.004, AUPRC 0.268 ± 0.013
LogReg | Trajectory + Static + Dynamic: AUROC 0.834 ± 0.003, AUPRC 0.288 ± 0.012
LogReg | Summary + Static + Dynamic: AUROC 0.848 ± 0.007, AUPRC 0.325 ± 0.024
LogReg | Trajectory + Summary + Static + Dynamic: AUROC 0.851 ± 0.007, AUPRC 0.328 ± 0.021
RF | Trajectory Only: AUROC 0.623 ± 0.006, AUPRC 0.100 ± 0.003
RF | Summary Stats Only: AUROC 0.647 ± 0.009, AUPRC 0.160 ± 0.010
RF | Trajectory + Summary Stats: AUROC 0.65

In [ ]:
summary = results_df.pivot_table(index='feature_set', columns='model', values='auroc_mean')
summary

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=results_df, x='feature_set', y='auroc_mean', hue='model')
plt.xticks(rotation=45, ha='right')
plt.title('AUROC by Feature Set and Model')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=results_df, x='feature_set', y='auprc_mean', hue='model')
plt.xticks(rotation=45, ha='right')
plt.title('AUPRC by Feature Set and Model')
plt.tight_layout()
plt.show()